# PPO policies by alpha: Vanilla vs MaxMin

Line plots: x=alpha, y=positiveness, conciseness (divisor=256), minority score, majority score.

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

USER_ID = "mxcui"
PPO_SEED = 42
PPO_PROPORTION = 0.2
DIVISOR = 256  # normalized_length = length / DIVISOR, conciseness = 1 - normalized_length

def _ppo_eval_id(method, alpha):
    return f"{USER_ID}/{method}-imdb-ppo-eval-prop{PPO_PROPORTION}-alpha{alpha}-seed{PPO_SEED}-kl0.1-gpt2"

def oracle_minority(pos, nl, alpha):
    return (1 - nl) * alpha + (1 - alpha) * pos

def oracle_majority(pos, nl, alpha):
    return (1 - nl) * (1 - alpha) + alpha * pos

ALPHAS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
results = {}  # (alpha, method) -> df

for alpha in ALPHAS:
    for method in ["vanilla", "maxmin"]:
        try:
            data = load_dataset(_ppo_eval_id(method, alpha))
            split = list(data.keys())[0] if hasattr(data, "keys") else "train"
            ds_ppo = data[split] if hasattr(data, "keys") else data
            df_ppo = ds_ppo.to_pandas()
        except Exception as e:
            print(f"Could not load {method} alpha={alpha}: {e}")
            continue

        if "positiveness" not in df_ppo.columns or "length" not in df_ppo.columns:
            import torch
            from transformers import AutoTokenizer, AutoModelForSequenceClassification
            model_name = "lvwerra/distilbert-imdb"
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            model.eval()
            pos_id = model.config.label2id["POSITIVE"]
            positiveness_list = []
            length_list = []
            batch_size = 64
            texts = df_ppo["completion"].astype(str).tolist()
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                enc = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
                with torch.no_grad():
                    out = model(**enc)
                probs = torch.softmax(out.logits, dim=-1)
                positiveness_list.extend(probs[:, pos_id].tolist())
                length_list.extend(enc["attention_mask"].sum(dim=1).tolist())
            df_ppo["positiveness"] = positiveness_list
            df_ppo["length"] = length_list

        df_ppo["normalized_length"] = np.clip(df_ppo["length"].astype(float) / DIVISOR, 0, 1)
        df_ppo["conciseness"] = 1 - df_ppo["normalized_length"]
        df_ppo["minority_score"] = oracle_minority(df_ppo["positiveness"], df_ppo["normalized_length"], alpha)
        df_ppo["majority_score"] = oracle_majority(df_ppo["positiveness"], df_ppo["normalized_length"], alpha)
        results[(alpha, method)] = df_ppo

if not results:
    print("No PPO eval data loaded. Skipping analysis.")
else:
    rows = []
    for (alpha, method), d in results.items():
        rows.append({
            "alpha": alpha,
            "method": method,
            "positiveness": d["positiveness"].mean(),
            "conciseness": d["conciseness"].mean(),
            "minority_score": d["minority_score"].mean(),
            "majority_score": d["majority_score"].mean(),
            "score_gap": (d["majority_score"] - d["minority_score"]).mean(),
        })
    df_agg = pd.DataFrame(rows)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    metrics = ["positiveness", "conciseness", "minority_score", "majority_score", "score_gap"]
    titles = ["Positiveness", "Conciseness (divisor=256)", "Minority score", "Majority score", "Group score gap (maj−min)"]
    for ax, metric, title in zip(axes, metrics, titles):
        for method in ["vanilla", "maxmin"]:
            sub = df_agg[(df_agg["method"] == method)].sort_values("alpha")
            if len(sub) > 0:
                ax.plot(sub["alpha"], sub[metric], "o-", label=method)
        ax.set_xlabel("Alpha")
        ax.set_ylabel(metric)
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
    axes[5].set_visible(False)  # hide unused subplot
    fig.suptitle(f"Vanilla vs MaxMin PPO by alpha (prop={PPO_PROPORTION}, seed={PPO_SEED})", y=1.02)
    plt.tight_layout()
    plt.show()



In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

USER_ID = "mxcui"
PPO_SEED = 42
PPO_PROPORTION = 0.2
DIVISOR = 256  # normalized_length = length / DIVISOR, conciseness = 1 - normalized_length

def _ppo_eval_id(method, alpha):
    return f"{USER_ID}/{method}-imdb-ppo-eval-prop{PPO_PROPORTION}-alpha{alpha}-seed{PPO_SEED}-kl0.1-gpt2"

def oracle_minority(pos, nl, alpha):
    return (1 - nl) * alpha + (1 - alpha) * pos

def oracle_majority(pos, nl, alpha):
    return (1 - nl) * (1 - alpha) + alpha * pos

ALPHAS = [0.5, 0.7, 1.0]
results = {}

for alpha in ALPHAS:
    for method in ["vanilla", "maxmin"]:
        try:
            data = load_dataset(_ppo_eval_id(method, alpha))
            split = list(data.keys())[0] if hasattr(data, "keys") else "train"
            df_ppo = data[split].to_pandas() if hasattr(data, "keys") else data.to_pandas()
        except Exception as e:
            print(f"Could not load {method} alpha={alpha}: {e}")
            continue
        if "positiveness" not in df_ppo.columns or "length" not in df_ppo.columns:
            import torch
            from transformers import AutoTokenizer, AutoModelForSequenceClassification
            tokenizer = AutoTokenizer.from_pretrained("lvwerra/distilbert-imdb")
            model = AutoModelForSequenceClassification.from_pretrained("lvwerra/distilbert-imdb")
            model.eval()
            pos_id = model.config.label2id["POSITIVE"]
            pl, ll = [], []
            texts = df_ppo["completion"].astype(str).tolist()
            for i in range(0, len(texts), 64):
                enc = tokenizer(texts[i:i+64], padding=True, truncation=True, max_length=512, return_tensors="pt")
                with torch.no_grad():
                    probs = torch.softmax(model(**enc).logits, dim=-1)
                pl.extend(probs[:, pos_id].tolist())
                ll.extend(enc["attention_mask"].sum(dim=1).tolist())
            df_ppo["positiveness"], df_ppo["length"] = pl, ll
        df_ppo["normalized_length"] = np.clip(df_ppo["length"].astype(float) / DIVISOR, 0, 1)
        df_ppo["conciseness"] = 1 - df_ppo["normalized_length"]
        df_ppo["minority_score"] = oracle_minority(df_ppo["positiveness"], df_ppo["normalized_length"], alpha)
        df_ppo["majority_score"] = oracle_majority(df_ppo["positiveness"], df_ppo["normalized_length"], alpha)
        df_ppo["score_gap"] = df_ppo["majority_score"] - df_ppo["minority_score"]
        results[(alpha, method)] = df_ppo

if not results:
    print("No PPO eval data loaded.")
else:
    rows = [{"alpha": a, "method": m,
             "positiveness": d["positiveness"].mean(),
             "conciseness": d["conciseness"].mean(),
             "minority_score": d["minority_score"].mean(),
             "majority_score": d["majority_score"].mean(),
             "score_gap": d["score_gap"].mean()}
            for (a, m), d in results.items()]
    df_agg = pd.DataFrame(rows)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for ax, metric, title in zip(axes,
        ["positiveness", "conciseness", "minority_score", "majority_score", "score_gap"],
        ["Positiveness", "Conciseness (div=256)", "Minority score", "Majority score", "Score gap (maj-min)"]):
        for method in ["vanilla", "maxmin"]:
            sub = df_agg[df_agg["method"] == method].sort_values("alpha")
            if len(sub): ax.plot(sub["alpha"], sub[metric], "o-", label=method)
        ax.set_xlabel("Alpha"); ax.set_ylabel(metric); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3)
    axes[5].set_visible(False)
    fig.suptitle(f"Vanilla vs MaxMin PPO by alpha (prop={PPO_PROPORTION}, seed={PPO_SEED})", y=1.02)
    plt.tight_layout()
    plt.show()



In [ ]:
import ast
import os
import pandas as pd
import matplotlib.pyplot as plt

WANDB_DIR = os.path.join(os.getcwd(), "wandb")

# MaxMin runs with kl_div=total (most recent complete run per alpha)
MAXMIN_RUNS = {
    0.5: "run-20260228_195714-g0nzcjeh",
    0.6: "run-20260301_154503-rdco48gz",
    0.7: "run-20260228_203638-a2fr5avj",
    0.8: "run-20260301_053917-lyc5jim9",
    0.9: "run-20260301_082946-r09pbjfb",
    1.0: "run-20260228_175121-irkolbmm",
}

def parse_group_scores(log_path):
    records = []
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if "group_selection/mean_reward_minority" not in line:
                continue
            try:
                d = ast.literal_eval(line)
                records.append({
                    "episode": int(d["episode"]),
                    "minority": float(d["group_selection/mean_reward_minority"]),
                    "majority": float(d["group_selection/mean_reward_majority"]),
                    "worst_group": d.get("group_selection/worst_group_name", ""),
                })
            except Exception:
                pass
    return pd.DataFrame(records)

SMOOTH_WINDOW = 20

alphas = sorted(MAXMIN_RUNS.keys())
ncols = 3
nrows = (len(alphas) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows), sharey=False)
axes = axes.flatten()

for ax, alpha in zip(axes, alphas):
    run_name = MAXMIN_RUNS[alpha]
    log_path = os.path.join(WANDB_DIR, run_name, "files", "output.log")
    if not os.path.isfile(log_path):
        ax.set_title(f"alpha={alpha}  [log not found]")
        ax.axis("off")
        continue

    df = parse_group_scores(log_path)
    if df.empty:
        ax.set_title(f"alpha={alpha}  [no data]")
        ax.axis("off")
        continue

    df = df.sort_values("episode").reset_index(drop=True)
    df["minority_smooth"] = df["minority"].rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean()
    df["majority_smooth"] = df["majority"].rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean()

    ax.plot(df["episode"], df["minority"], alpha=0.2, color="tab:blue")
    ax.plot(df["episode"], df["minority_smooth"], color="tab:blue", linewidth=2, label="Minority")
    ax.plot(df["episode"], df["majority"], alpha=0.2, color="tab:orange")
    ax.plot(df["episode"], df["majority_smooth"], color="tab:orange", linewidth=2, label="Majority")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_title(f"MaxMin PPO  alpha={alpha}")
    ax.set_xlabel("Training episode")
    ax.set_ylabel("Oracle score (batch mean)")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

for ax in axes[len(alphas):]:
    ax.set_visible(False)

fig.suptitle(
    f"Minority vs Majority oracle scores during MaxMin PPO training\n"
    f"(kl_div=total, kl_coef=0.1, smoothing window={SMOOTH_WINDOW})",
    y=1.01
)
plt.tight_layout()
plt.show()